In [1]:
import os
import time
import psutil
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# 1. Loading data and sample 100k rows to run fast because it was taking too long before
df = pd.read_parquet("feature_table_v1.parquet")
if len(df) > 100_000:
    df = df.sample(n=100_000, random_state=42)

target_col = "sales"
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# 2. Converting datetimes to numbers
for col in X.select_dtypes(include=['datetime64', 'datetime64[ns]']).columns:
    X[f"{col}_year"] = X[col].dt.year
    X[f"{col}_month"] = X[col].dt.month
    X[f"{col}_day"] = X[col].dt.day
    X.drop(columns=[col], inplace=True)

# 3. Converting categories and booleans to clean integers
for col in X.select_dtypes(include=['object', 'category']).columns:
    X[col] = X[col].astype('category').cat.codes.astype('int32')

for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype('int8')

# 4. Splitting
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Benchmark function
def benchmark(name, model):
    process = psutil.Process(os.getpid())
    mem_start = process.memory_info().rss / (1024 * 1024)
    t_start = time.perf_counter()

    model.fit(X_train, y_train)

    duration = time.perf_counter() - t_start
    mem_used = max(0.0, (process.memory_info().rss / (1024 * 1024)) - mem_start)

    preds = model.predict(X_val)
    return {
        "Model": name,
        "RMSE": round(root_mean_squared_error(y_val, preds), 4),
        "MAE": round(mean_absolute_error(y_val, preds), 4),
        "Training Time (s)": round(duration, 2),
        "Peak Memory (MB)": round(mem_used, 2)
    }

# 6. Training all 4 models
models = [
    ("Random Forest", RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)),
    ("XGBoost", XGBRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)),
    ("LightGBM", LGBMRegressor(n_estimators=50, max_depth=6, random_state=42, verbose=-1, n_jobs=-1)),
    ("CatBoost", CatBoostRegressor(iterations=50, depth=6, random_state=42, verbose=0, thread_count=-1))
]

results = [benchmark(name, model) for name, model in models]

# 7. Print final deliverable table
print(pd.DataFrame(results).to_markdown(index=False))

| Model         |    RMSE |     MAE |   Training Time (s) |   Peak Memory (MB) |
|:--------------|--------:|--------:|--------------------:|-------------------:|
| Random Forest | 465.979 | 112.33  |                1.76 |              15.04 |
| XGBoost       | 470.212 | 148.646 |                0.25 |             105.91 |
| LightGBM      | 542.349 | 178.962 |                0.13 |               2.59 |
| CatBoost      | 484.363 | 170.24  |                0.4  |              11.75 |


In [3]:
# in terms of speed vs memory, the lightGBM delivered the fastest training time and the smallest memory footprint, making it
# source-efficient framework on tabullar structures

In [4]:
# in baseline accuracy winner is the random forest which produced the strongest initial results without any tuning, demonstrating the stability of bagging
#when deep trees are averaged together


In [6]:
# while boosting algorithms xgboots, lightgbm and catboost finished training rapidly in under half a second, they require fine tuning such as 
# optimizing learning rates, tree depth and early stopping to surpass random forest out of box accuracy


In [7]:
# framework selection : bagging (random forest ) serves like ideal, zero configuration baseline resilient to noise whereas gradient boosting remains the preffered
#tool for pushing performance boundaries on structred data once properly tuned
]